In [0]:
%sql
-- Create a table with sensitive employee data
CREATE OR REPLACE TABLE workspace.default.employee_sensitive (
    id          INT,
    name        STRING,
    department  STRING,
    salary      DOUBLE,
    ssn         STRING,   -- sensitive! Social Security Number
    email       STRING
);

-- Insert sample data
INSERT INTO workspace.default.employee_sensitive VALUES
(1, 'Alice',   'Engineering', 95000.00, '123-45-6789', 'alice@company.com'),
(2, 'Bob',     'Finance',     85000.00, '234-56-7890', 'bob@company.com'),
(3, 'Charlie', 'HR',          75000.00, '345-67-8901', 'charlie@company.com'),
(4, 'Diana',   'Engineering', 92000.00, '456-78-9012', 'diana@company.com');

SELECT * FROM workspace.default.employee_sensitive;

In [0]:
%sql
-- GRANT = give access to someone

-- Give SELECT access to a user
GRANT SELECT ON TABLE workspace.default.employee_sensitive 
TO `nikita.upadhayay@gmail.com`;

-- Give SELECT access to all users
GRANT SELECT ON TABLE workspace.default.employee_sensitive 
TO `account users`;

-- Give MODIFY access (INSERT, UPDATE, DELETE)
GRANT MODIFY ON TABLE workspace.default.employee_sensitive 
TO `nikita.upadhayay@gmail.com`;

-- Give access at SCHEMA level (all tables in schema)
GRANT SELECT ON SCHEMA workspace.default 
TO `nikita.upadhayay@gmail.com`;

-- Check what permissions exist
SHOW GRANTS ON TABLE workspace.default.employee_sensitive;

In [0]:
%sql
-- REVOKE = take away access

-- Remove MODIFY access
REVOKE MODIFY ON TABLE workspace.default.employee_sensitive 
FROM `nikita.upadhayay@gmail.com`;

-- Check permissions again
SHOW GRANTS ON TABLE workspace.default.employee_sensitive;

In [0]:
%sql
-- Column masking = hide sensitive column values
-- SSN should show as *** for non-HR users

-- Create a masking function
CREATE OR REPLACE FUNCTION mask_ssn(ssn STRING)
RETURN CASE 
    WHEN is_member('hr_group') THEN ssn  
    ELSE '***-**-****'                    
END;

-- Apply mask to SSN column
ALTER TABLE workspace.default.employee_sensitive
ALTER COLUMN ssn 
SET MASK mask_ssn;

-- Query the table
SELECT * FROM workspace.default.employee_sensitive;

In [0]:
%sql
-- Row level security = hide entire rows
-- Engineering team sees only Engineering rows
-- Finance team sees only Finance rows

-- Create row filter function
CREATE OR REPLACE FUNCTION filter_by_department(department STRING)
RETURN CASE
    WHEN is_member('hr_group')          THEN TRUE  -- HR sees all
    WHEN is_member('engineering_group') 
         AND department = 'Engineering' THEN TRUE  -- Eng sees Eng only
    WHEN is_member('finance_group')     
         AND department = 'Finance'     THEN TRUE  -- Finance sees Finance only
    ELSE FALSE
END;

-- Apply row filter to table
ALTER TABLE workspace.default.employee_sensitive
SET ROW FILTER filter_by_department ON (department);

-- Query the table
SELECT * FROM workspace.default.employee_sensitive;